In [0]:
import json
import os
from datetime import datetime
from databricks.sdk import WorkspaceClient

# Credenziali / Path Volume Databricks
PATH_VOLUME_BASE = "/Volumes/workspace_file_loading/staging/factory_details_raw_data/"

# Generazione timestamp per partizionamento e nome unico
current_time = datetime.now()
data_caricamento = current_time.strftime("%Y-%m-%d")
timestamp_str = current_time.strftime("%Y%m%d_%H%M%S")

# 1. Definiamo la cartella partizionata per data e il file univoco
folder_path = f"{PATH_VOLUME_BASE}data_loading={data_caricamento}/"
file_name = f"vendite_AZ-TECH-01_{timestamp_str}.json"
full_volume_path = f"{folder_path}{file_name}"

# Esempio dati del giorno
data = {
  "id_azienda": "AZ-TECH-01",
  "azienda": "TechCorp Solutions",
  "estrazione_timestamp": "2026-09-22T10:00:00Z",
  "ordini": [
    {
      "ordine_id": "ORD-2026-1001",
      "data_ordine": "2026-09-22T08:30:00Z",
      "canale_vendita": "Online Store",
      "importo_totale": 1580.50,
      "valuta": "EUR",
      "cliente": [
        {
          "cliente_id": "CLI-9821",
          "nome": "Marco",
          "cognome": "Rossi",
          "email": "marco.rossi@example.com",
          "segmento": "B2B",
          "indirizzo_citta": "Milano",
          "indirizzo_paese": "Italia"
        }
      ],
      "dettagli_ordine": [
        {
          "riga_id": "RIGA-1001-1",
          "quantita": 1,
          "prezzo_unitario": 1200.00,
          "sconto_applicato": 0.05,
          "prodotto": [
            {
              "prodotto_id": "PROD-TECH-01",
              "nome_prodotto": "Laptop Pro 16",
              "categoria": "Informatica",
              "settore_id": "SET-ELECTRONICS",
              "nome_settore": "Elettronica & IT",
              "responsabile_settore": "Laura Bianchi"
            }
          ]
        },
        {
          "riga_id": "RIGA-1001-2",
          "quantita": 1,
          "prezzo_unitario": 440.50,
          "sconto_applicato": 0.00,
          "prodotto": [
            {
              "prodotto_id": "PROD-TECH-02",
              "nome_prodotto": "Monitor 4K 27\"",
              "categoria": "Informatica",
              "settore_id": "SET-ELECTRONICS",
              "nome_settore": "Elettronica & IT",
              "responsabile_settore": "Laura Bianchi"
            }
          ]
        }
      ],
      "pagamenti": [
        {
          "transazione_id": "TRX-883921",
          "metodo_pagamento": "Carta di Credito",
          "circuito": "Visa",
          "stato_pagamento": "COMPLETATO",
          "importo_pagato": 1580.50
        }
      ]
    }
  ]
}

print(f"Salvataggio file giornaliero in corso su Databricks Volume...")
print(f"Destinazione: {full_volume_path}")

# 2. Caricamento su Databricks Volume tramite SDK
w = WorkspaceClient()

try:
    # Convertiamo il dizionario JSON in stream di byte
    json_bytes = json.dumps(data, indent=2).encode('utf-8')
    
    # Upload sul Volume (crea automaticamente le cartelle se non esistono)
    w.files.upload(full_volume_path, json_bytes, overwrite=True)
    print("File del giorno caricato con successo nel Volume!")

except Exception as e:
    print(f"Errore durante il caricamento: {str(e)}")